<a href="https://colab.research.google.com/github/Harsimran726/Research-Documentation/blob/main/Micro_Grad_without_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [255]:
import math

class Value:
  def __init__(self,data,childern=(), label='',_op=''):
    self.data = data
    self.label= label
    self._prev = set(childern)
    self._op = _op
    self._backward = lambda : None
    self.grad = 0

  def __repr__(self):
    return f"Value(Data={self.data})"

  def __neg__(self):
    return self * -1

  def __radd__(self,other):
    return self + other

  def __sub__(self,other):
    return self + (-other)

  def __truediv__(self,other):
    return self * other**-1

  def __rmul__(self,other):
    return self * other

  def __add__(self,other):
    other = other if isinstance(other,Value) else Value(other)
    out = Value(self.data + other.data , (self,other), '+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out



  def __mul__(self,other):
    other = other if isinstance(other,Value) else Value(other)
    out = Value(self.data * other.data , (self,other), "*")
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad

    out._backward = _backward
    return out

  def tanh(self):
    x= self.data
    t = (math.exp(2*x)-1)/(math.exp(2*x)+1)
    out = Value(t,(self,),'tanh')
    def _backward():
      self.grad += (1.0 - t**2) * out.grad
    out._backward = _backward
    return out

  def __pow__(self,other):
    assert isinstance(other,(int,float))
    out = Value(self.data ** other,(self,),f'**{other}') # Changed f'**other' to f'**{other}' for clarity
    def _backward():
      self.grad += (other * ( self.data ** (other -1))) * out.grad # Fixed typo: self.grade -> self.grad, out.grade -> out.grad
    out._backward = _backward

    return out

  def exp(self):
    x = self.data
    out = Value(math.exp(x),(self,),'exp')
    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward
    return out

  def backward(self):
    topo = []
    visited = set()

    def buildtopo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          buildtopo(child)
        topo.append(v)
    buildtopo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()



In [256]:
"""
MLP: that contain three parts : Neuron , Layer : MLP (Multi Layer Perceptron )
Neuron: xi , wi , bias
Layer: Neruson(nin,nout)
MLP: Layer = (layer(sz[i],sz[i+1]) for _ in range(len(nout)))
   - call with for i in range:...
   -
"""

'\nMLP: that contain three parts : Neuron , Layer : MLP (Multi Layer Perceptron )\nNeuron: xi , wi , bias\nLayer: Neruson(nin,nout)\nMLP: Layer = (layer(sz[i],sz[i+1]) for _ in range(len(nout)))\n   - call with for i in range:...\n   -\n'

In [257]:
import random
class Neuron:
  def __init__(self,nin):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]  # weights for each neuron of the leyer
    self.b = Value(random.uniform(-1,1))
  # we create the neuron in each layer and then calcualtes ..
  def __call__(self,x):
    # Fixed: changed '+' to '*' for weighted sum (dot product)
    act = sum((wi * xi for wi,xi in zip(self.w,x)),self.b)
    out = act.tanh()
    return out

  def parameters(self):
    return self.w + [self.b]

class Layer:
  def __init__(self,nin,nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]
  # it help us to create the number of layers in our perceptron
  def __call__(self,x):
    out = [n(x) for n in self.neurons]
    return out[0] if len(out)==1 else out

  def parameters(self):
    return [p for neurons in self.neurons for p in neurons.parameters()]

class MLP:
  def __init__(self,nin,nout):
    sz = [nin] + nout
    self.layer = [Layer(sz[i],sz[i+1]) for i in range(len(nout))]
  # here we combine the layers in to Multi layer perceptron
  def __call__(self,x):
    temp = 0
    for layers in self.layer:
      # print(f"layers: {layers}")
      x = layers(x)
      temp+=1
    return x

  def parameters(self):
    # Fixed typo: self.layers -> self.layer
    return [p for layers in self.layer for p in layers.parameters()]

In [258]:
xs = [
    [1.0,3.2,-1.3],
    [2.1,4.2,-12],
    [3.2,41,3.2]
]
x = [1.0,2.1,-23.2]
target_value = [1.0,-1.2,-12.4]
neuron = MLP(3,[4,3,1])
neuron(x)
# [neuron(x) for x in xs]

Value(Data=-0.9107788524328495)

In [260]:
def run_nn():
  xs = [
        [1.9, -1.2, 3.0],
        [0.2, -1.3, -1.2],
        [0.2, 3.3, 0.3],
  ]

  # Scaled targets to be within the [-1, 1] range of tanh
  target_value = [0.2, 0.8, -0.5]

  # Initialize a Multi-Layer Perceptron: 3 inputs, two hidden layers of 4, and 1 output
  model = MLP(3, [4, 4, 1])

  learning_rate = 0.05

  for k in range(20): # Increased epochs slightly to see better convergence
    # Forward pass
    ypred = [model(x) for x in xs]

    # Calculate Mean Squared Error Loss
    loss = sum((yout - ygt)**2 for ygt, yout in zip(target_value, ypred))

    # Zero gradients before backward pass
    for p in model.parameters():
      p.grad = 0.0

    # Backward pass
    loss.backward()

    # Update parameters (Gradient DESCENT)
    for p in model.parameters():
      p.data -= learning_rate * p.grad

    print(f"Epoch {k} | Loss: {loss.data:.4f}")

# Call the function after everything is defined
run_nn()

Epoch 0 | Loss: 1.9757
Epoch 1 | Loss: 1.6563
Epoch 2 | Loss: 1.1737
Epoch 3 | Loss: 0.5364
Epoch 4 | Loss: 0.2079
Epoch 5 | Loss: 0.1136
Epoch 6 | Loss: 0.0727
Epoch 7 | Loss: 0.0501
Epoch 8 | Loss: 0.0362
Epoch 9 | Loss: 0.0270
Epoch 10 | Loss: 0.0206
Epoch 11 | Loss: 0.0159
Epoch 12 | Loss: 0.0126
Epoch 13 | Loss: 0.0100
Epoch 14 | Loss: 0.0081
Epoch 15 | Loss: 0.0066
Epoch 16 | Loss: 0.0054
Epoch 17 | Loss: 0.0045
Epoch 18 | Loss: 0.0037
Epoch 19 | Loss: 0.0031
